#set up

In [1]:
#set up
import pandas as pd
import numpy as np

In [2]:
from google.colab import drive
drive.mount('/content/drive')
data = ('/content/drive/My Drive/Colab Notebooks/SocialMediaPosts/MatchTextLength1204/01OrginalData/')
result = ('/content/drive/My Drive/Colab Notebooks/SocialMediaPosts/MatchTextLength1204/02DataForAnalysis/')

Mounted at /content/drive


# Clinical groups: merge the uttrances of the same speakers

## Episodicity_He_Hinzen_dataset

Episodicity_He_Hinzen_dataset: filter out the data based on the value in ‘diag’ column. If ‘diag’= ‘ModerateAD’ or ‘MildAD’, the data belongs to AD_clinical group; If ‘diag’= ‘Control’, the data belongs to Health_clinical group

### read the data and filter AD and health group


In [4]:
He = pd.read_csv(data + 'Episodicity_He_Hinzen_dataset.csv')
He.head(2)

,PAR,age,gender,mmse,edu_yrs,diag,original_text,clean_text,annotation,category
0,dementia-001-2,59,male,11,14,INV,you see going on in the picture . 0_1808,you see going on in the picture .,you see going on in the picture .,Meta
1,dementia-001-2,59,male,11,14,INV,tell me all the action . 1808_3609,tell me all the action .,tell me all the action .,Meta


In [5]:
# Display the unique values themselves
print("Unique types in 'diag':", He['diag'].unique())

Unique types in 'diag': ['INV' 'ModerateAD' 'MildAD' 'Control']


In [6]:
# Filter the DataFrame to keep only rows where 'diag' is 'ModerateAD' or 'MildAD'
He_AD = He[He['diag'].isin(['ModerateAD', 'MildAD'])]

print(f"Number of rows: {He_AD.shape[0]}")
print(f"Number of columns: {He_AD.shape[1]}")
print(He_AD['diag'].unique())

Number of rows: 1789
Number of columns: 10
['ModerateAD' 'MildAD']


In [8]:
# Filter the DataFrame to keep only rows where 'diag' is 'control'
He_health = He[He['diag'].isin(['Control'])]

print(f"Number of rows: {He_health.shape[0]}")
print(f"Number of columns: {He_health.shape[1]}")
print(He_health['diag'].unique())

Number of rows: 1020
Number of columns: 10
['Control']


### merge text group by speakers

The data was preprocessed and cleaned. Each participant’s utterances were originally split into separate rows in the ‘content’ column. We combined all utterances/contents spoken by the same participant into a single row/cell, grouped by participant ID.  

#### He_AD

(1) for He_AD

In [9]:
He_AD.columns

Index(['PAR', 'age', 'gender', 'mmse', 'edu_yrs', 'diag', 'original_text',
       'clean_text', 'annotation', 'category'],
      dtype='object')

In [10]:
# Ensure 'clean_text' is treated as a string and handle NaNs
He_AD['clean_text'] = He_AD['clean_text'].fillna('').astype(str)

<ipython-input-10-95e403bff0ba>:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  He_AD['clean_text'] = He_AD['clean_text'].fillna('').astype(str)


In [11]:
# Group by 'PAR' and aggregate
# - Concatenate 'clean_text' using ' '.join
# - Keep the first occurrence of other columns
grouped_He_AD = He_AD.groupby('PAR').agg({
    'clean_text': ' '.join,
    **{col: 'first' for col in He_AD.columns if col not in ['PAR', 'clean_text']}
}).reset_index()

In [12]:
grouped_He_AD

,PAR,clean_text,age,gender,mmse,edu_yrs,diag,original_text,annotation,category
0,dementia-001-2,mhm . there's a young boy going in a cookie ja...,59,male,11,14,ModerateAD,mhm . 3609_4282,mhm .,Interj
1,dementia-003-0,here's a cookie jar . and the lid is off the c...,56,male,20,17,MildAD,here's a cookie jar . 0_8778,here's a cookie jar .,NONEP
2,dementia-005-2,okay he's falling off a chair . she's running ...,55,male,19,9,MildAD,okay he's fallin(g) off a chair [: stool] [* s...,{okay}_OtherItj he's falling off a chair .,EP
3,dementia-007-3,well the girl is telling the boy to get the co...,75,female,15,14,ModerateAD,well the girl is telling the boy to get the co...,{well}_OtherFiller the girl is telling the boy...,EP
4,dementia-010-0,oh boy . wowie the boy's going up on a cookiej...,66,male,20,12,MildAD,oh boy . [+ exc] 2194_3657,oh boy .,Interj
...,...,...,...,...,...,...,...,...,...,...
123,dementia-695-0,yeah . everything's that's not according to Ho...,83,male,16,12,MildAD,yeah . [+ exc] 1563_2100,yeah .,Interj
124,dementia-698-0,well there's a cookie jar there with the lid o...,78,female,12,8,ModerateAD,well there's a cookie jar there with the lid o...,{well}_OtherFiller there's a cookie jar there ...,NONEP
125,dementia-702-0,you want me to tell you ? okay the boy's getti...,65,female,20,12,MildAD,you want me to tell you ? [+ exc] 5361_6358,you want me to tell you ?,Meta
126,dementia-703-0,well here's the child reaching up but he's on ...,73,female,13,10,ModerateAD,&uh well here's the child reaching up,{well}_OtherFiller here's the child {reaching ...,NONEP


#### He_health

(2) for He_health

In [13]:
He_health.columns

Index(['PAR', 'age', 'gender', 'mmse', 'edu_yrs', 'diag', 'original_text',
       'clean_text', 'annotation', 'category'],
      dtype='object')

In [14]:
He_health['clean_text'] = He_health['clean_text'].fillna('').astype(str)

<ipython-input-14-1a34b1398c23>:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  He_health['clean_text'] = He_health['clean_text'].fillna('').astype(str)


In [15]:
grouped_He_health = He_health.groupby('PAR').agg({
    'clean_text': ' '.join,
    **{col: 'first' for col in He_health.columns if col not in ['PAR', 'clean_text']}
}).reset_index()

In [16]:
grouped_He_health.head()

,PAR,clean_text,age,gender,mmse,edu_yrs,diag,original_text,annotation,category
0,control-002-0,the scene is in the kitchen . the mother is wi...,58,female,30,16,Control,the scene is in the kitchen . 0_5703,the scene is in the kitchen .,NONEP
1,control-013-0,somebody's getting cookies out of the cookie j...,62,female,30,12,Control,somebody's getting cookies out_of the cookie j...,somebody's getting cookies out of the cookie j...,EP
2,control-015-1,okay . a little boy is stepping on a stool tha...,67,female,30,15,Control,okay . [+ exc] 0_2478,okay .,Interj
3,control-017-4,are you ready ? well the sink is overflowing ....,71,female,30,15,Control,are you ready ? [+ exc] 0_2397,are you ready ?,Meta
4,control-021-1,okay . the mother's washing the dishes and the...,74,female,30,12,Control,okay . [+ exc] 2291_3332,okay .,Interj


In [17]:
grouped_He_AD.shape

(128, 10)

In [18]:
grouped_He_health.shape

(70, 10)

In [19]:
print("Unique types in grouped_He_AD 'diag':", grouped_He_AD['diag'].unique())

Unique types in grouped_He_AD 'diag': ['ModerateAD' 'MildAD']


In [20]:
print("Unique types in grouped_He_health 'diag':", grouped_He_health['diag'].unique())

Unique types in grouped_He_health 'diag': ['Control']


## protocol_baycrest_delaware_mci

The data was preprocessed and cleaned. Each participant’s utterances were originally split into separate rows in the ‘content’ column. We combined all utterances/contents spoken by the same participant into a single row/cell, grouped by participant ID.  

### Delaware_AD

In [21]:
Delaware_AD = pd.read_csv(data + 'protocol_baycrest_delaware_mci_110624.csv')
Delaware_AD = Delaware_AD.drop(columns=['Unnamed: 0']) #drop the Unnamed: 0 column
Delaware_AD

,speaker,content,content_len,content_clean,content_clean_len,content_semi_clean,content_semi_len,task,PID,language,corpus,age,sex,CogDis,participant,text_clean
0,PAR,right . 7839_8129,1.0,right,1.0,right,1.0,NaN,11312/a-00064235-1,eng,Baycrest,74;00.,female,MCI,Baycrest12257,right.
1,PAR,oh_gosh . 25280_26230,1.0,oh gosh,2.0,oh gosh,2.0,Important_Event,11312/a-00064235-1,eng,Baycrest,74;00.,female,MCI,Baycrest12257,oh gosh.
2,PAR,I don't really know . 26541_27821,4.0,I don't really know,4.0,I don't really know,4.0,Important_Event,11312/a-00064235-1,eng,Baycrest,74;00.,female,MCI,Baycrest12257,I don't really know.
3,PAR,I guess maybe when we moved to Port_Stanley . ...,8.0,I guess maybe when we moved to Port Stanley,9.0,I guess maybe when we moved to Port Stanley,9.0,Important_Event,11312/a-00064235-1,eng,Baycrest,74;00.,female,MCI,Baycrest12257,I guess maybe when we moved to Port Stanley.
4,PAR,&-um I'm Japanese . 33032_35222,3.0,um I'm Japanese,3.0,um I'm Japanese,3.0,Important_Event,11312/a-00064235-1,eng,Baycrest,74;00.,female,MCI,Baycrest12257,um I'm Japanese.
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
7687,PAR,and spread the peanut butter . 760345_763385,5.0,and spread the peanut butter,5.0,and spread the peanut butter,5.0,Sandwich,11312/a-00078226-1,eng,Delaware,88;00.,female,MCI,23-1,and spread the peanut butter.
7688,PAR,and then dot it with raisins . 763845_765825,6.0,and then dot it with raisins,6.0,and then dot it with raisins,6.0,Sandwich,11312/a-00078226-1,eng,Delaware,88;00.,female,MCI,23-1,and then dot it with raisins.
7689,PAR,and cover it up . 767965_769625,4.0,and cover it up,4.0,and cover it up,4.0,Sandwich,11312/a-00078226-1,eng,Delaware,88;00.,female,MCI,23-1,and cover it up.
7690,PAR,and slice it diagonally . 769625_770765,4.0,and slice it diagonally,4.0,and slice it diagonally,4.0,Sandwich,11312/a-00078226-1,eng,Delaware,88;00.,female,MCI,23-1,and slice it diagonally.


In [22]:
Delaware_AD.columns

Index(['speaker', 'content', 'content_len', 'content_clean',
       'content_clean_len', 'content_semi_clean', 'content_semi_len', 'task',
       'PID', 'language', 'corpus', 'age', 'sex', 'CogDis', 'participant',
       'text_clean'],
      dtype='object')

In [24]:
# Display the unique values themselves
print("Unique types in 'diag':", Delaware_AD['CogDis'].unique())

Unique types in 'diag': ['MCI' 'AD' nan]


In [25]:
Delaware_AD = Delaware_AD[Delaware_AD['CogDis'] == 'MCI']
Delaware_AD

,speaker,content,content_len,content_clean,content_clean_len,content_semi_clean,content_semi_len,task,PID,language,corpus,age,sex,CogDis,participant,text_clean
0,PAR,right . 7839_8129,1.0,right,1.0,right,1.0,NaN,11312/a-00064235-1,eng,Baycrest,74;00.,female,MCI,Baycrest12257,right.
1,PAR,oh_gosh . 25280_26230,1.0,oh gosh,2.0,oh gosh,2.0,Important_Event,11312/a-00064235-1,eng,Baycrest,74;00.,female,MCI,Baycrest12257,oh gosh.
2,PAR,I don't really know . 26541_27821,4.0,I don't really know,4.0,I don't really know,4.0,Important_Event,11312/a-00064235-1,eng,Baycrest,74;00.,female,MCI,Baycrest12257,I don't really know.
3,PAR,I guess maybe when we moved to Port_Stanley . ...,8.0,I guess maybe when we moved to Port Stanley,9.0,I guess maybe when we moved to Port Stanley,9.0,Important_Event,11312/a-00064235-1,eng,Baycrest,74;00.,female,MCI,Baycrest12257,I guess maybe when we moved to Port Stanley.
4,PAR,&-um I'm Japanese . 33032_35222,3.0,um I'm Japanese,3.0,um I'm Japanese,3.0,Important_Event,11312/a-00064235-1,eng,Baycrest,74;00.,female,MCI,Baycrest12257,um I'm Japanese.
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
7687,PAR,and spread the peanut butter . 760345_763385,5.0,and spread the peanut butter,5.0,and spread the peanut butter,5.0,Sandwich,11312/a-00078226-1,eng,Delaware,88;00.,female,MCI,23-1,and spread the peanut butter.
7688,PAR,and then dot it with raisins . 763845_765825,6.0,and then dot it with raisins,6.0,and then dot it with raisins,6.0,Sandwich,11312/a-00078226-1,eng,Delaware,88;00.,female,MCI,23-1,and then dot it with raisins.
7689,PAR,and cover it up . 767965_769625,4.0,and cover it up,4.0,and cover it up,4.0,Sandwich,11312/a-00078226-1,eng,Delaware,88;00.,female,MCI,23-1,and cover it up.
7690,PAR,and slice it diagonally . 769625_770765,4.0,and slice it diagonally,4.0,and slice it diagonally,4.0,Sandwich,11312/a-00078226-1,eng,Delaware,88;00.,female,MCI,23-1,and slice it diagonally.


In [26]:
# Display the unique values themselves
print("Unique types in 'diag':", Delaware_AD['CogDis'].unique())

Unique types in 'diag': ['MCI']


In [37]:
Delaware_AD['text_clean'] = Delaware_AD['text_clean'].fillna('').astype(str)

grouped_Delaware_AD = Delaware_AD.groupby('participant').agg({
    'text_clean': ' '.join,
    **{col: 'first' for col in Delaware_AD.columns if col not in ['participant','text_clean']}
}).reset_index()

#grouped_Delaware_AD = Delaware_AD.groupby('participant').agg({...}).reset_index()

<ipython-input-37-b7c51aa269f9>:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  Delaware_AD['text_clean'] = Delaware_AD['text_clean'].fillna('').astype(str)


In [38]:
grouped_Delaware_AD

,participant,text_clean,speaker,content,content_len,content_clean,content_clean_len,content_semi_clean,content_semi_len,task,PID,language,corpus,age,sex,CogDis
0,01-1,oh my heavens. the child is on a stool that is...,PAR,oh my heavens . [+ exc] 4870_5600,4.0,oh my heavens,3.0,oh my heavens,3.0,Cookie,11312/a-00048788-1,eng,Delaware,84;00.,female,MCI
1,05-1,okay. looks like a kitchen. mom uh washing or ...,PAR,okay . [+ exc] 5890_6600,2.0,okay,1.0,okay,1.0,Cookie,11312/a-00055315-1,eng,Delaware,79;00.,male,MCI
2,06-1,um the kids um horsing around um trying to get...,PAR,&-um the kids &-um horsing around &-um trying ...,13.0,um the kids um horsing around um trying to get...,13.0,um the kids um horsing around um trying to get...,13.0,Cookie,11312/a-00056209-1,eng,Delaware,81;00.,female,MCI
3,09-1,well there are at least two accidents about to...,PAR,well ‡ there are at least two accidents about ...,13.0,well there are at least two accidents about to...,13.0,well there are at least two accidents about to...,13.0,Cookie,11312/a-00078219-1,eng,Delaware,73;00.,male,MCI
4,10-1,yes. okay. well there's a little boy getting c...,PAR,yes .,1.0,yes,1.0,yes,1.0,Cookie,11312/a-00078220-1,eng,Delaware,78;00.,female,MCI
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
62,Baycrest11976,alright. you're going. uh the reason I'm laugh...,PAR,alright . 6646_6956,1.0,alright,1.0,alright,1.0,Important_Event,11312/a-00064234-1,eng,Baycrest,88;00.,male,MCI
63,Baycrest12257,right. oh gosh. I don't really know. I guess m...,PAR,right . 7839_8129,1.0,right,1.0,right,1.0,Important_Event,11312/a-00064235-1,eng,Baycrest,74;00.,female,MCI
64,Baycrest12813,thinking back tell you a story. I have an inte...,PAR,"thinking back , tell you a story . 20810_22810",6.0,thinking back tell you a story,6.0,thinking back tell you a story,6.0,Important_Event,11312/a-00064236-1,eng,Baycrest,66;00.,male,MCI
65,Baycrest7352,can you repeat the question? okay well as I si...,PAR,can you repeat the question ? 29210_30400,5.0,can you repeat the question,5.0,can you repeat the question,5.0,Important_Event,11312/a-00064239-1,eng,Baycrest,71;00.,female,MCI


In [39]:
print("Unique types in grouped_Delaware_AD:", grouped_Delaware_AD['CogDis'].unique())

Unique types in grouped_Delaware_AD: ['MCI']


### Delaware_health

In [40]:
Delaware_health = pd.read_csv(data + 'protocol_baycrest_delaware_control_110624.csv')
Delaware_health = Delaware_health.drop(columns=['Unnamed: 0']) #drop the Unnamed: 0 column
Delaware_health.head(2)

,speaker,content,content_len,content_clean,content_clean_len,content_semi_clean,content_semi_len,task,PID,language,corpus,age,sex,CogDis,participant,text_clean
0,PAR,yeah . 15875_16375,1.0,yeah,1.0,yeah,1.0,NaN,11312/a-00065467-1,eng,Delaware,84;00.,female,Control,97-1,yeah.
1,PAR,well ‡ it looks like the window's open . 2559...,7.0,well it looks like the window's open,7.0,well it looks like the window's open,7.0,Cookie,11312/a-00065467-1,eng,Delaware,84;00.,female,Control,97-1,well it looks like the window's open.


In [41]:
# Display the unique values themselves
print("Unique types in 'diag':", Delaware_health['CogDis'].unique())

Unique types in 'diag': ['Control']


In [42]:
Delaware_health['text_clean'] = Delaware_health['text_clean'].fillna('').astype(str)

grouped_Delaware_health = Delaware_health.groupby('participant').agg({
    'text_clean': ' '.join,
    **{col: 'first' for col in Delaware_health.columns if col not in ['participant', 'text_clean']}
}).reset_index()

In [44]:
grouped_Delaware_health

,participant,text_clean,speaker,content,content_len,content_clean,content_clean_len,content_semi_clean,content_semi_len,task,PID,language,corpus,age,sex,CogDis
0,02-1,well um it's sort of dangerous with the little...,PAR,(..) well ‡ &-um it's sort_of dangerous with t...,25.0,well um it's sort of dangerous with the little...,27.0,well um it's sort of dangerous with the little...,27.0,Cookie,11312/a-00078230-1,eng,Delaware,62;00.,female,Control
1,03-1,kids are stealing cookies out of the cookie ja...,PAR,kids are stealing cookies out_of the cookie ja...,9.0,kids are stealing cookies out of the cookie ja...,10.0,kids are stealing cookies out of the cookie ja...,10.0,Cookie,11312/a-00048790-1,eng,Delaware,63;00.,female,Control
2,04-1,yes. okay. a a boy is on a stepstool trying to...,PAR,yes . [+ exc] 7870_8210,2.0,yes,1.0,yes,1.0,Cookie,11312/a-00048791-1,eng,Delaware,72;00.,female,Control
3,07-1,well I see the young boy looking for the cooki...,PAR,well ‡ I see the young boy looking for the coo...,11.0,well I see the young boy looking for the cooki...,11.0,well I see the young boy looking for the cooki...,11.0,Cookie,11312/a-00055316-1,eng,Delaware,81;00.,female,Control
4,08-1,oh it's not a pretty picture. a young man uh r...,PAR,oh ‡ it's not a pretty picture . 17220_19760,6.0,oh it's not a pretty picture,6.0,oh it's not a pretty picture,6.0,Cookie,11312/a-00055317-1,eng,Delaware,68;00.,male,Control
5,101-1,hm woman washing dishes. water spilling on the...,PAR,&-hm woman washing dishes . 45595_47935,4.0,hm woman washing dishes,4.0,hm woman washing dishes,4.0,Cookie,11312/a-00082447-0,eng,Delaware,60;00.,female,Control
6,12-1,oh my goodness. okay. um daddy was outside pla...,PAR,oh_my_goodness . [+ exc] 12940_14160,2.0,oh my goodness,3.0,oh my goodness,3.0,Cat,11312/a-00055318-1,eng,Delaware,82;00.,female,Control
7,13-1,kitchen disaster. okay um mom or mom figure is...,PAR,kitchen disaster . 3010_4110,2.0,kitchen disaster,2.0,kitchen disaster,2.0,Cookie,11312/a-00055319-1,eng,Delaware,76;00.,female,Control
8,24-1,um there's a woman standing at the sink drying...,PAR,"&-um there's a woman standing at the sink , dr...",11.0,um there's a woman standing at the sink drying...,11.0,um there's a woman standing at the sink drying...,11.0,Cookie,11312/a-00055324-1,eng,Delaware,66;00.,female,Control
9,25-1,well that boy is greedy cause he wants to go i...,PAR,well ‡ that boy is greedy (be)cause he wants t...,13.0,well that boy is greedy because he wants to go...,13.0,well that boy is greedy cause he wants to go i...,13.0,Cookie,11312/a-00055325-1,eng,Delaware,70;00.,male,Control


## combine data from different dataset

### AD_clinical

In [45]:
#Add a new column named 'corpus' with the value 'He_Hinzen' for every row, so we can know the source of the text
grouped_He_AD['corpus'] = 'He_Hinzen'

# Add a new column named 'index' starting from 0
#grouped_He_AD['index'] = np.arange(len(grouped_He_AD))

grouped_He_AD.head(2)

,PAR,clean_text,age,gender,mmse,edu_yrs,diag,original_text,annotation,category,corpus
0,dementia-001-2,mhm . there's a young boy going in a cookie ja...,59,male,11,14,ModerateAD,mhm . 3609_4282,mhm .,Interj,He_Hinzen
1,dementia-003-0,here's a cookie jar . and the lid is off the c...,56,male,20,17,MildAD,here's a cookie jar . 0_8778,here's a cookie jar .,NONEP,He_Hinzen


In [46]:
grouped_He_AD.shape

(128, 11)

In [47]:
grouped_Delaware_AD.head(2)

,participant,text_clean,speaker,content,content_len,content_clean,content_clean_len,content_semi_clean,content_semi_len,task,PID,language,corpus,age,sex,CogDis
0,01-1,oh my heavens. the child is on a stool that is...,PAR,oh my heavens . [+ exc] 4870_5600,4.0,oh my heavens,3.0,oh my heavens,3.0,Cookie,11312/a-00048788-1,eng,Delaware,84;00.,female,MCI
1,05-1,okay. looks like a kitchen. mom uh washing or ...,PAR,okay . [+ exc] 5890_6600,2.0,okay,1.0,okay,1.0,Cookie,11312/a-00055315-1,eng,Delaware,79;00.,male,MCI


In [48]:
grouped_Delaware_AD.shape

(67, 16)

In [49]:
grouped_He_AD.columns

Index(['PAR', 'clean_text', 'age', 'gender', 'mmse', 'edu_yrs', 'diag',
       'original_text', 'annotation', 'category', 'corpus'],
      dtype='object')

In [50]:
grouped_Delaware_AD.columns

Index(['participant', 'text_clean', 'speaker', 'content', 'content_len',
       'content_clean', 'content_clean_len', 'content_semi_clean',
       'content_semi_len', 'task', 'PID', 'language', 'corpus', 'age', 'sex',
       'CogDis'],
      dtype='object')

The common column in both dataframe, but they have different name, so we need to rename the column.
- participant/PAR
- age
- sex/gender
- clean_text/content_semi_clean
- diag/CogDis


In [51]:
# rename the columns of grouped_Delaware_AD
# Define a dictionary mapping old column names to new column names
rename_dict = {
    'participant': 'PAR',
    'CogDis': 'diag',
    'sex': 'gender',
    'text_clean': 'clean_text'}

grouped_Delaware_AD = grouped_Delaware_AD.rename(columns=rename_dict)
grouped_Delaware_AD.head(2)

,PAR,clean_text,speaker,content,content_len,content_clean,content_clean_len,content_semi_clean,content_semi_len,task,PID,language,corpus,age,gender,diag
0,01-1,oh my heavens. the child is on a stool that is...,PAR,oh my heavens . [+ exc] 4870_5600,4.0,oh my heavens,3.0,oh my heavens,3.0,Cookie,11312/a-00048788-1,eng,Delaware,84;00.,female,MCI
1,05-1,okay. looks like a kitchen. mom uh washing or ...,PAR,okay . [+ exc] 5890_6600,2.0,okay,1.0,okay,1.0,Cookie,11312/a-00055315-1,eng,Delaware,79;00.,male,MCI


In [52]:
#Find common columns between the two DataFrames
common_columns = list(set(grouped_He_AD).intersection(set(grouped_Delaware_AD)))
common_columns

['age', 'corpus', 'diag', 'clean_text', 'PAR', 'gender']

In [53]:
columns = [ 'PAR','corpus', 'diag', 'age','gender','clean_text']

In [54]:
# Concatenate the DataFrames using only the common columns
AD_clinical = pd.concat([grouped_He_AD[columns],
                         grouped_Delaware_AD[columns]],
                        ignore_index=True)


In [55]:
AD_clinical

,PAR,corpus,diag,age,gender,clean_text
0,dementia-001-2,He_Hinzen,ModerateAD,59,male,mhm . there's a young boy going in a cookie ja...
1,dementia-003-0,He_Hinzen,MildAD,56,male,here's a cookie jar . and the lid is off the c...
2,dementia-005-2,He_Hinzen,MildAD,55,male,okay he's falling off a chair . she's running ...
3,dementia-007-3,He_Hinzen,ModerateAD,75,female,well the girl is telling the boy to get the co...
4,dementia-010-0,He_Hinzen,MildAD,66,male,oh boy . wowie the boy's going up on a cookiej...
...,...,...,...,...,...,...
190,Baycrest11976,Baycrest,MCI,88;00.,male,alright. you're going. uh the reason I'm laugh...
191,Baycrest12257,Baycrest,MCI,74;00.,female,right. oh gosh. I don't really know. I guess m...
192,Baycrest12813,Baycrest,MCI,66;00.,male,thinking back tell you a story. I have an inte...
193,Baycrest7352,Baycrest,MCI,71;00.,female,can you repeat the question? okay well as I si...


In [56]:
grouped_He_AD.shape

(128, 11)

In [57]:
grouped_Delaware_AD.shape

(67, 16)

In [58]:
AD_clinical.shape

(195, 6)

In [59]:
# save the results to a new cvs. file
AD_clinical.to_csv(result + 'AD_clinical_clean_data.csv', index=False)

### Health_clinical

In [60]:
#Add a new column named 'corpus' with the value 'He_Hinzen' for every row, so we can know the source of the text
grouped_He_health['corpus'] = 'He_Hinzen'

# Add a new column named 'index' starting from 0
# grouped_He_health['index'] = np.arange(len(grouped_He_health))

grouped_He_health.head(2)

,PAR,clean_text,age,gender,mmse,edu_yrs,diag,original_text,annotation,category,corpus
0,control-002-0,the scene is in the kitchen . the mother is wi...,58,female,30,16,Control,the scene is in the kitchen . 0_5703,the scene is in the kitchen .,NONEP,He_Hinzen
1,control-013-0,somebody's getting cookies out of the cookie j...,62,female,30,12,Control,somebody's getting cookies out_of the cookie j...,somebody's getting cookies out of the cookie j...,EP,He_Hinzen


In [61]:
grouped_Delaware_health.head(2)

,participant,text_clean,speaker,content,content_len,content_clean,content_clean_len,content_semi_clean,content_semi_len,task,PID,language,corpus,age,sex,CogDis
0,02-1,well um it's sort of dangerous with the little...,PAR,(..) well ‡ &-um it's sort_of dangerous with t...,25.0,well um it's sort of dangerous with the little...,27.0,well um it's sort of dangerous with the little...,27.0,Cookie,11312/a-00078230-1,eng,Delaware,62;00.,female,Control
1,03-1,kids are stealing cookies out of the cookie ja...,PAR,kids are stealing cookies out_of the cookie ja...,9.0,kids are stealing cookies out of the cookie ja...,10.0,kids are stealing cookies out of the cookie ja...,10.0,Cookie,11312/a-00048790-1,eng,Delaware,63;00.,female,Control


In [65]:
# rename the columns of grouped_Delaware_health
# Define a dictionary mapping old column names to new column names
rename_dict = {
    'participant': 'PAR',
    'CogDis': 'diag',
    'sex': 'gender',
    'content_semi_clean': 'clean_text'}

grouped_Delaware_health = grouped_Delaware_health.rename(columns=rename_dict)
grouped_Delaware_health.head(2)

,PAR,text_clean,speaker,content,content_len,content_clean,content_clean_len,clean_text,content_semi_len,task,PID,language,corpus,age,gender,diag
0,02-1,well um it's sort of dangerous with the little...,PAR,(..) well ‡ &-um it's sort_of dangerous with t...,25.0,well um it's sort of dangerous with the little...,27.0,well um it's sort of dangerous with the little...,27.0,Cookie,11312/a-00078230-1,eng,Delaware,62;00.,female,Control
1,03-1,kids are stealing cookies out of the cookie ja...,PAR,kids are stealing cookies out_of the cookie ja...,9.0,kids are stealing cookies out of the cookie ja...,10.0,kids are stealing cookies out of the cookie ja...,10.0,Cookie,11312/a-00048790-1,eng,Delaware,63;00.,female,Control


In [64]:
# Concatenate the DataFrames using only the common columns
Health_clinical = pd.concat([grouped_He_health[columns],
                         grouped_Delaware_health[columns]],
                        ignore_index=True)

Health_clinical

,PAR,corpus,diag,age,gender,clean_text
0,control-002-0,He_Hinzen,Control,58,female,the scene is in the kitchen . the mother is wi...
1,control-013-0,He_Hinzen,Control,62,female,somebody's getting cookies out of the cookie j...
2,control-015-1,He_Hinzen,Control,67,female,okay . a little boy is stepping on a stool tha...
3,control-017-4,He_Hinzen,Control,71,female,are you ready ? well the sink is overflowing ....
4,control-021-1,He_Hinzen,Control,74,female,okay . the mother's washing the dishes and the...
...,...,...,...,...,...,...
99,94-1,Delaware,Control,68;00.,female,yep
100,96-1,Delaware,Control,61;00.,female,yep
101,97-1,Delaware,Control,84;00.,female,yeah
102,98-1,Delaware,Control,63;00.,female,okay


In [66]:
grouped_He_health.shape

(70, 11)

In [67]:
grouped_Delaware_health.shape

(34, 16)

In [68]:
Health_clinical.shape

(104, 6)

In [69]:
# save the results to a new cvs. file
Health_clinical.to_csv(result + 'Health_clinical_clean_data.csv', index=False)